## Imports

In [ ]:
# Add project root to Python path
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
print(f"Added to Python path: {project_root}")

In [ ]:
import pandas as pd
import numpy as np

## Favorita 1-store SKU-level meta-sets 31/07

In [ ]:
favorita_normalized = pd.read_pickle(f"{project_root}/data/favorita/processed_consecutive/favorita_cleaned_naSalesffill_2y.pkl")

In [ ]:
# Create store-specific, item-based Favorita meta-sets
np.random.seed(42)

STORE_NBR = 47
TRAIN_END = 334
VAL_START, VAL_END = 251, 362
TEST_START, TEST_END = 279, 390

VAL_ITEM_COUNT = 500
TEST_ITEM_COUNT = 1000
TRAIN_ITEM_COUNT = 50

item_col = 'item_nbr' if 'item_nbr' in favorita_normalized.columns else 'traj_id'

# Restrict to one store (robust to numeric/string formatting of store_nbr)
store_num = pd.to_numeric(favorita_normalized['store_nbr'], errors='coerce')
store_mask = store_num.eq(float(STORE_NBR))
if not store_mask.any():
    store_mask = favorita_normalized['store_nbr'].astype(str).str.replace('.0', '', regex=False).eq(str(STORE_NBR))
store_df = favorita_normalized[store_mask].copy()

# Build period-specific views first so sampled items are guaranteed to exist in their target ranges
train_df_full = store_df[(store_df['time_idx'] >= 0) & (store_df['time_idx'] <= TRAIN_END)].copy()
val_df_full = store_df[(store_df['time_idx'] >= VAL_START) & (store_df['time_idx'] <= VAL_END)].copy()
test_df_full = store_df[(store_df['time_idx'] >= TEST_START) & (store_df['time_idx'] <= TEST_END)].copy()

train_pool_period = set(train_df_full[item_col].unique())
val_pool = set(val_df_full[item_col].unique())
test_pool = set(test_df_full[item_col].unique())

# Disjoint split: test reserved first, then val from the remainder, then train.
meta_test_items = np.random.choice(sorted(test_pool), size=TEST_ITEM_COUNT, replace=False)
remaining_for_val = sorted(val_pool - set(meta_test_items))
meta_val_items = np.random.choice(remaining_for_val, size=VAL_ITEM_COUNT, replace=False)

excluded_items = set(meta_val_items) | set(meta_test_items)
train_pool = sorted(train_pool_period - excluded_items)
shuffled_train_pool = np.random.permutation(train_pool).tolist()
meta_train_items = shuffled_train_pool[:TRAIN_ITEM_COUNT]

meta_val_store47 = val_df_full[val_df_full[item_col].isin(meta_val_items)].copy()
meta_test_store47 = test_df_full[test_df_full[item_col].isin(meta_test_items)].copy()
meta_train_store47 = train_df_full[train_df_full[item_col].isin(meta_train_items)].copy()

# Save
output_dir = project_root / 'data' / 'favorita' / 'meta_sets'
output_dir.mkdir(parents=True, exist_ok=True)

val_path = output_dir / f'Seed42_store{STORE_NBR}_itemBased_meta_val_{VAL_ITEM_COUNT}items_3107.pkl'
test_path = output_dir / f'Seed42_store{STORE_NBR}_itemBased_meta_test_{TEST_ITEM_COUNT}items_3107.pkl'
train_path = output_dir / f'Seed42_store{STORE_NBR}_itemBased_meta_train_cumulative_{TRAIN_ITEM_COUNT}items_3107.pkl'
meta_val_store47.to_pickle(val_path)
meta_test_store47.to_pickle(test_path)
meta_train_store47.to_pickle(train_path)

print('Created store-specific item-based meta sets')
print(f'  Store: {STORE_NBR}')
print(f'  Val:   items={len(meta_val_items)}, rows={len(meta_val_store47)}, time_idx=[{VAL_START}, {VAL_END}]')
print(f'  Test:  items={len(meta_test_items)}, rows={len(meta_test_store47)}, time_idx=[{TEST_START}, {TEST_END}]')
print(f'  Train: items={len(meta_train_items)}, rows={len(meta_train_store47)}, time_idx=[0, {TRAIN_END}]')
print(f'  Saved to: {output_dir}')